In [59]:
# Подключаем библиотеки

import re
from dataclasses import dataclass
import parser as p

# Действия
ACTIONS  = {"выбрать", "показать", "найти", "удалить"}

# Число

QUANT = {"все", "один"}

# Объект, как цель

OBJECTS  = {"машины", "автомобили", "авто", "автомобиль", "машину"}

# Харпактеристики и поля базы данных

FIELDS   = {"цвет", "марка", "модель", "год", "пробег", "цена", "кузов",
            "коробка", "привод", "мощность", "владелец"}

# Операторы сравнения

OPS = {"равен", "равна", "равно", "больше", "меньше"}

# Ключевые слова связки

KEYWORDS = {"и", "или", "не", "если", "из", "имеет"}

# Определяем, что слово прилагательное и не обязательно перед ним писать равно и т.д.

ADJ = {
    "красный": ("цвет", "красный"), "красная": ("цвет", "красный"),
    "красное": ("цвет", "красный"), "белый": ("цвет", "белый"),
    "белая": ("цвет", "белый"), "чёрный": ("цвет", "чёрный"),
    "чёрная": ("цвет", "чёрный"),
}

# Смотрим как делить, как строка или слово

TOKEN_RE = re.compile(r'"[^"]*"|[^\s"]+')

In [60]:
# Создаем класс токена, хранит тип, значение и позицию, для определения, где произошла ошибка

@dataclass
class Token:
    type: str
    value: str
    pos: int

In [61]:
# Разделяем на токе, токеном может выступать число строка или слово отдельно
# Внедрена обработка кавычек, чтобы слово в кавычках считалось одним токеном, если не хотим получить ошибку в дальнейшем предлагаю учитывать в программе строгий порядок слов, чтобы пользователь вводил кавычки если название автомобиля сложное из 2-3 слов.
def tokenize(text):

    # Фиксируем токены

    tokens = []

    for m in TOKEN_RE.finditer(text):
        v = m.group()

        # Стоит заметить, что у нас 3 вида токенов STRING, NUMBER, WORD

        if v.startswith('"'):
            tokens.append(Token("STRING", v.strip('"'), m.start()))
        else:
            v = v.strip(".,!?;:")
            if v:
                tokens.append(Token("NUMBER" if v.isdigit() else "WORD", v, m.start()))

    # Позволяет сказатью что это конец набора токенов для предложения

    tokens.append(Token("EOF", "", len(text)))

    return tokens

In [62]:
# Главная функция

def analyze(query: str):

    print("=" * 70)
    print("Запрос:", query)
    print("-" * 70)

    # Разбираем строку на токены

    tokens = tokenize(query)
    if not tokens:
        print("Результат: ОШИБКА")
        print("Пустой запрос")
        return

    print(tokens)

    # Производим парсинг строим деревья

    pars = p.Parser(tokens)

In [63]:
# Успешные запросы

# 1) одинарные снаружи, двойные внутри
analyze('найти машины из таблицы 1 если марка "равна toyota" и год больше 2020')

# 2) экранирование
analyze("найти машины из таблицы 1 если марка \"равна toyota\" и год больше 2020")

# 3) тройные кавычки
analyze("""найти машины из таблицы 1 если марка "равна toyota" и год больше 2020""")

Запрос: найти машины из таблицы 1 если марка "равна toyota" и год больше 2020
----------------------------------------------------------------------
[Token(type='WORD', value='найти', pos=0), Token(type='WORD', value='машины', pos=6), Token(type='WORD', value='из', pos=13), Token(type='WORD', value='таблицы', pos=16), Token(type='NUMBER', value='1', pos=24), Token(type='WORD', value='если', pos=26), Token(type='WORD', value='марка', pos=31), Token(type='STRING', value='равна toyota', pos=37), Token(type='WORD', value='и', pos=52), Token(type='WORD', value='год', pos=54), Token(type='WORD', value='больше', pos=58), Token(type='NUMBER', value='2020', pos=65), Token(type='EOF', value='', pos=69)]
Запрос: найти машины из таблицы 1 если марка "равна toyota" и год больше 2020
----------------------------------------------------------------------
[Token(type='WORD', value='найти', pos=0), Token(type='WORD', value='машины', pos=6), Token(type='WORD', value='из', pos=13), Token(type='WORD', val